# Module 02 — μ-parameterization| | ||---|---|| **Input** | the FM4NPP source tree || **Algorithm** | width-dependent init + width-scaled learning rates || **Output** | a working `--mup / --no-mup` switch for module 03 || **Visualization** | learning rate vs width; the optimal-LR shift when μP is off |**What you'll learn:** what μP is for, exactly where it lives in this codebase, the twoindependent reasons it currently does nothing, and how to turn it on and off.**Runtime:** ~2 minutes. No GPU, no data.

## 1. The problem μP solvesYou tune a learning rate on a small model. You scale the model up 30×. Your carefullytuned LR is now wrong — often catastrophically. So you re-tune at the larger size, whichcosts far more, and you re-tune again at the next size.μ-parameterization (Yang & Hu, 2021) reparameterizes initialization and per-parameterlearning rates so that **the optimal LR stops moving as width grows**. Tune once, cheaply,at small width; transfer to the large model.For a foundation model spanning 0.34M → 188M parameters, this is not a nicety. It's thedifference between one LR sweep and six.FM4NPP's released ladder:| paper | width `Nu` | `d_state` `Nx` | params ||---|---|---|---|| m3 | 256 | 16 | 5.3M || m4 | 512 | 32 | 21M || m5 | 1024 | 64 | 84M || m6 | 1536 | 96 | 175M |Note `Nx = Nu/16` **exactly**. Textbook μP holds the state/head dimension fixed while widthgrows. Here it doesn't. Keep that in mind — it changes what the scaling exponents mean.

In [ ]:
import os, sys, inspect, mathimport torchimport numpy as npimport matplotlib.pyplot as pltFM4NPP_ROOT = os.environ.get('FM4NPP_ROOT')if FM4NPP_ROOT and FM4NPP_ROOT not in sys.path:    sys.path.insert(0, FM4NPP_ROOT)import mup   # this module, in 02_mu_parameterization/LADDER = [('m3', 256, 16), ('m4', 512, 32), ('m5', 1024, 64), ('m6', 1536, 96)]print('fm4npp root:', FM4NPP_ROOT or '(not set -- source common/paths.sh)')

## 2. Where μP actually is in this codebaseHere is the first surprise. **The pretraining script has no μP at all.** Its own docstringsays so:```Mamba1 Training Script - Simplified Mamba (no μ-transfer, FP32)...2. Standard AdamW optimizer```and its initialization is width-independent:```pythonwith torch.no_grad():    for name, param in self.model.named_parameters():        if "norm.weight" in name:  init.ones_(param)        elif "bias" in name:       init.zeros_(param)```The μP-shaped code lives entirely in the **downstream** trainers. Let's look at it.

In [ ]:
# The width-dependent initializer, straight from the repo.path = os.path.join(FM4NPP_ROOT, 'train/downstream/track_finding_trainer.py')src = open(path).read()start = src.index('def initialize_mamba2(model, d_state, embed_dim)')print(src[start:start + 900])

So the init rule is$$\mathrm{std}(\texttt{lin\_B}) = \sqrt{N_x/N_u}, \qquad  \mathrm{std}(\texttt{lin\_C}) = \sqrt{1/(N_u N_x)}$$**This is only partial μP.** It touches `lin_B`, `lin_C`, norms and biases. `in_proj`,`out_proj`, `conv1d`, the embedder and the output layer keep PyTorch's default`U(-1/√fan_in, 1/√fan_in)`, which is *not* what μP prescribes. Worth knowing before youtrust the transfer.Now the learning rates.

In [ ]:
i = src.index('params_a   = []')print(src[i - 120: i + 1250])

Four groups, three of them width-scaled:| group | learning rate ||---|---|| `A_log` | `base_lr · Nu` || `lin_B` | `base_lr · Nx/√Nu` || `lin_C` | `base_lr · √Nu/Nx` || everything else | `base_lr` |Let's tabulate that across the released ladder.

In [ ]:
base_lr = 2e-5rows = []for paper, Nu, Nx in LADDER:    t = mup.mup_lr_table(Nu, Nx, base_lr)    rows.append((paper, Nu, Nx, t['A_log'], t['lin_B'], t['lin_C'], t['other']))print(f'{"paper":<6}{"Nu":>6}{"Nx":>5}{"A_log":>12}{"lin_B":>12}{"lin_C":>12}{"other":>12}')for r in rows:    print(f'{r[0]:<6}{r[1]:>6}{r[2]:>5}' + ''.join(f'{x:>12.3e}' for x in r[3:]))

Two things to notice.**`A_log` gets an enormous rate.** At m6 it is `2e-5 × 1536 = 0.031` — over 1500× the baserate. `A_log` parameterizes the state-decay matrix through a log, so its natural scalegenuinely differs from a weight matrix, but this is still a large multiplier to acceptwithout a coordinate check.**At m3 the B and C rules vanish.** `Nx/√Nu = 16/16 = 1` and `√Nu/Nx = 16/16 = 1`. At width256 μP is invisible except through `A_log`. If you test LR transfer only at m3, you will seenothing — which is exactly why the sweep below uses widths 128 and 512 as well.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))widths = np.array([w for _, w, _ in LADDER])for key, style in (('A_log', 'o-'), ('lin_B', 's-'), ('lin_C', '^-'), ('other', 'd--')):    ax.plot(widths, [mup.mup_lr_table(w, x, base_lr)[key] for _, w, x in LADDER],            style, label=key, lw=1.8, ms=6)ax.set_xscale('log', base=2); ax.set_yscale('log')ax.set_xticks(widths); ax.set_xticklabels([f'{p}\n{w}' for p, w, _ in LADDER])ax.set_xlabel('model (paper name / width $N_u$)'); ax.set_ylabel('learning rate')ax.set_title('μP learning rate per parameter group, base_lr = 2e-5')ax.grid(alpha=.3, which='both'); ax.legend()plt.tight_layout(); plt.show()

## 3. Why none of this currently does anythingNow the part that matters. The μP code above is present, correct-looking, and **inert** —for two independent reasons.### Death #1 — the scheduler flattens every group`CosineAnnealingWarmupRestarts.__init__` calls `init_lr()`:```pythondef init_lr(self):    self.base_lrs = []    for param_group in self.optimizer.param_groups:        param_group['lr'] = self.min_lr        # <-- every group, same scalar        self.base_lrs.append(self.min_lr)```The trainer constructs that scheduler **three lines after** building the four groups. Run it:

In [ ]:
before, after = mup.demonstrate_scheduler_nullification(Nu=1536, Nx=96, base_lr=2e-5)print('m6, four μP groups:')print(f'  before scheduler : {[f"{x:.3e}" for x in before]}')print(f'  after  scheduler : {[f"{x:.3e}" for x in after]}')print()print('all four identical:', len(set(after)) == 1)print(f'A_log lost a factor of {before[0]/after[0]:,.0f}x')

Every bit of width scaling, gone. No warning, no error. The optimizer still *has* fourgroups — they just all carry the same learning rate, so the grouping is decorative.### Death #2 — the init is overwritten by the checkpointIn the downstream path, `initialize_mamba2(self.model, Nx, Nu)` runs in `__init__`. Then, ahundred lines later:```pythonself.restore_checkpoint(self.params.pretrained_ckpt)```which overwrites **every backbone weight** with the pretrained values. The width-dependentinit only survives in a from-scratch run.So in normal downstream use, both halves of μP are no-ops.### The evidence that the real μP script existedThe downstream trainer does this when loading a pretrained checkpoint:```pythonself.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])```into the **four-group** optimizer. The shipped pretraining script saves a **one-group**state. PyTorch raises `ValueError: loaded state dict has a different number of parametergroups` on that mismatch — so whatever produced the released checkpoints saved four groups.The μP pretraining script existed. It is just not in the public repository.

## 4. Turning it on, and turning it off`mup.py` in this directory reconstructs the rule and makes it actually take effect.`attach_scheduler()` keeps the scheduler's *shape* while preserving the per-group ratios.

In [ ]:
Nu, Nx = 1536, 96dummy = [torch.nn.Parameter(torch.zeros(1)) for _ in range(4)]lrs = mup.mup_lr_table(Nu, Nx, 2e-5)opt = torch.optim.AdamW([{'params': [dummy[i]], 'lr': lrs[k]}                         for i, k in enumerate(('A_log', 'lin_B', 'lin_C', 'other'))])sched = mup.attach_scheduler(opt, total_steps=1000, max_lr=2e-4,                             min_lr=2e-5, warmup_steps=100)

In [ ]:
hist = {k: [] for k in ('A_log', 'lin_B', 'lin_C', 'other')}for _ in range(1000):    for k, g in zip(hist, opt.param_groups):        hist[k].append(g['lr'])    sched.step()fig, axes = plt.subplots(1, 2, figsize=(11, 4))for k, v in hist.items():    axes[0].plot(v, label=k, lw=1.6)axes[0].set_yscale('log'); axes[0].set_xlabel('step'); axes[0].set_ylabel('lr')axes[0].set_title('μP preserved through the schedule'); axes[0].legend(); axes[0].grid(alpha=.3)axes[1].plot(np.array(hist['A_log']) / np.array(hist['other']), lw=1.8, color='crimson')axes[1].axhline(Nu, ls='--', c='k', lw=1, label=f'$N_u$ = {Nu}')axes[1].set_xlabel('step'); axes[1].set_ylabel('A_log / other')axes[1].set_title('ratio is constant, as μP requires'); axes[1].legend(); axes[1].grid(alpha=.3)plt.tight_layout(); plt.show()print(f"peak 'other' lr: {max(hist['other']):.3e}   (max_lr = 2.000e-04)")print(f"ratio A_log/other: constant at {hist['A_log'][0]/hist['other'][0]:.1f} = Nu")

### The switch```pythonapply_mup_init(model, Nu, Nx, enabled=args.mup)opt = build_mup_optimizer(model, Nu, Nx, base_lr, enabled=args.mup)```With `enabled=False` you get exactly the shipped behaviour: one parameter group at`base_lr`, and PyTorch-default init with no width dependence. That is the `--no-mup` arm ofmodule 03.

In [ ]:
class Toy(torch.nn.Module):    # Minimal stand-in carrying the parameter names the rules match on.    def __init__(self, Nu, Nx):        super().__init__()        self.A_log = torch.nn.Parameter(torch.zeros(Nu // 16))        self.lin_B = torch.nn.Linear(Nu, Nx, bias=False)        self.lin_C = torch.nn.Linear(Nx, Nu, bias=False)        self.out   = torch.nn.Linear(Nu, Nu)for enabled in (True, False):    print('=' * 62)    m = Toy(1536, 96)    mup.apply_mup_init(m, 1536, 96, enabled=enabled)    o = mup.build_mup_optimizer(m, 1536, 96, base_lr=2e-5, enabled=enabled)    print(f'  lin_B std after init: {m.lin_B.weight.std().item():.5f}')    print(f'  param groups: {len(o.param_groups)}')

## 5. What breaking μP costs youModule 03 runs the payoff experiment: the same 200 steps at widths 128 / 256 / 512, sweptover learning rate, with μP on and off.**With μP**, the loss-vs-LR curves should have their minima at roughly the same LR at everywidth — that's LR transfer, the whole point.**Without μP**, the minima drift with width. An LR tuned at width 128 is then wrong at 512,and progressively more wrong as you scale up. That is the inconsistent scaling this moduleis about.```bashcd ../03_pretrainingpython lr_transfer_sweep.py --widths 128,256,512 --steps 200```That sweep is 18 short runs. On one A100 it's about 25 minutes.---## Summary1. The pretraining script in the public repo has **no μP**.2. The downstream trainers contain μP-shaped code that is **inert twice over** — the   scheduler flattens the learning rates, and the checkpoint load overwrites the init.3. The real μP pretraining script is **not published**; the four-group optimizer state the   downstream code expects is the fingerprint it left behind.4. `mup.py` reconstructs the rule so you can study it, and gives module 03 an on/off switch.**Next:** [Module 03 — Pretraining](../03_pretraining/)